## **Projeto:** Merca Data Platform
### **Squad:** 2 | Camada Gold
### Origem e Destino
| Item | Valor |

| Origem | squad2/silver/* (Delta Lake) |

| Destino | squad2/gold/* (Delta Lake) |

| Checkpoint | squad2/control/gold/control_file.json |

| Dependencia | Silver das 3 tabelas com status CONCLUIDO |

### Visoes Geradas
| Tabela | Descricao |

| gold_produtos_ativos | Catalogo limpo de produtos ativos com preco valido |

| gold_itens_pedido_validos | Itens sem desconto abusivo prontos para faturamento |

| gold_categorias_raiz | Arvore de categorias validada |

### Regras de Negocio Aplicadas
| Codigo | Regra |

| PRD-R04 | Produto ativo com preco_lista = 0 descartado |

| ITP-R06 | Item com desconto maior que preco_unitario descartado |

| CAT-R04 | Numero de categorias raiz nao pode ser zero |


In [0]:
%pip install deltalake

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
import logging
import pandas as pd
from deltalake import DeltaTable, write_deltalake
from datetime import datetime

logging.getLogger("azure").setLevel(logging.WARNING)

CAMADA = "gold"

path_silver_cat = get_delta_path("silver", "ecommerce_categorias")
path_silver_itp = get_delta_path("silver", "ecommerce_itens_pedido")
path_silver_prd = get_delta_path("silver", "ecommerce_produtos")

path_gold_cat = get_delta_path(CAMADA, "gold_categorias_raiz")
path_gold_itp = get_delta_path(CAMADA, "gold_itens_pedido_validos")
path_gold_prd = get_delta_path(CAMADA, "gold_produtos_ativos")

inicio = log_inicio("feat_squad2_" + CAMADA)
log.info("Camada : " + CAMADA)

In [0]:
def processar_gold_produtos() -> bool:
    try:
        log.info("Processando gold_produtos_ativos...")

        df = DeltaTable(
            path_silver_prd, storage_options=get_storage_options()
        ).to_pandas()

        total = len(df)
        log.info("Linhas lidas da Silver produtos: " + str(total))

        # PRD-R04 Negocio: produto ativo com preco_lista = 0
        antes = len(df)
        df = df[~((df["is_ativo"] == True) & (df["preco_lista"] <= 0))].copy()
        log.info("PRD-R04: " + str(antes - len(df)) + " produto(s) ativo(s) com preco zero descartado(s).")

        # Seleciona colunas relevantes
        colunas = [
            "sku", "nome_produto", "id_categoria", "preco_lista",
            "unidade_medida", "nome_marca", "is_ativo",
            "bronze_ingested_at", "silver_processed_at"
        ]
        colunas_presentes = [c for c in colunas if c in df.columns]
        df = df[colunas_presentes].reset_index(drop=True).copy()

        df["gold_processed_at"] = datetime.now()

        for col in df.columns:
            if pd.api.types.is_datetime64_any_dtype(df[col]):
                df[col] = df[col].dt.tz_localize(None)

        # Grava na Gold Delta
        write_deltalake(
            table_or_uri    = path_gold_prd,
            data            = df,
            mode            = "overwrite",
            storage_options = get_storage_options(),
            schema_mode     = "overwrite"
        )
        log.info("gold_produtos_ativos Delta gravado: " + str(len(df)) + " linhas.")

        # Grava no SQL Server
        df_spark = spark.createDataFrame(df)
        sucesso_sql = gravar_sql(df_spark, "gold_produtos_ativos")
        log.info("gold_produtos_ativos SQL Server: " + str("OK" if sucesso_sql else "FALHOU"))

        return True

    except Exception as e:
        log.error("Erro em gold_produtos_ativos: " + str(e))
        return False

In [0]:
def processar_gold_itens_pedido() -> bool:
    try:
        log.info("Processando gold_itens_pedido_validos...")

        df = DeltaTable(
            path_silver_itp, storage_options=get_storage_options()
        ).to_pandas()

        total = len(df)
        log.info("Linhas lidas da Silver itens_pedido: " + str(total))

        # ITP-R06 Negocio: desconto maior que preco_unitario
        antes = len(df)
        df = df[df["desconto_aplicado"] <= df["preco_unitario"]].copy()
        log.info("ITP-R06: " + str(antes - len(df)) + " item(ns) com desconto abusivo descartado(s).")

        # Calcula valor final do item
        df["valor_total_item"] = (
            df["quantidade"] * df["preco_unitario"] - df["desconto_aplicado"]
        ).round(2)

        # Seleciona colunas relevantes
        colunas = [
            "id_item_pedido", "id_pedido", "sku", "quantidade",
            "preco_unitario", "desconto_aplicado", "valor_total_item",
            "bronze_ingested_at", "silver_processed_at"
        ]
        colunas_presentes = [c for c in colunas if c in df.columns]
        df = df[colunas_presentes].reset_index(drop=True).copy()

        df["gold_processed_at"] = datetime.now()

        for col in df.columns:
            if pd.api.types.is_datetime64_any_dtype(df[col]):
                df[col] = df[col].dt.tz_localize(None)

        # Grava na Gold Delta
        write_deltalake(
            table_or_uri    = path_gold_itp,
            data            = df,
            mode            = "overwrite",
            storage_options = get_storage_options(),
            schema_mode     = "overwrite"
        )
        log.info("gold_itens_pedido_validos Delta gravado: " + str(len(df)) + " linhas.")

        # Grava no SQL Server
        df_spark = spark.createDataFrame(df)
        sucesso_sql = gravar_sql(df_spark, "gold_itens_pedido_validos")
        log.info("gold_itens_pedido_validos SQL Server: " + str("OK" if sucesso_sql else "FALHOU"))

        return True

    except Exception as e:
        log.error("Erro em gold_itens_pedido_validos: " + str(e))
        return False

In [0]:
def processar_gold_categorias() -> bool:
    try:
        log.info("Processando gold_categorias_raiz...")

        df = DeltaTable(
            path_silver_cat, storage_options=get_storage_options()
        ).to_pandas()

        total = len(df)
        log.info("Linhas lidas da Silver categorias: " + str(total))

        # CAT-R04 Negocio: numero de categorias raiz nao pode ser zero
        col_pai = "id_categoria_pai" if "id_categoria_pai" in df.columns else None

        if col_pai:
            df_raiz = df[df[col_pai].isna()].copy()
            if len(df_raiz) == 0:
                raise ValueError("CAT-R04: Nenhuma categoria raiz encontrada - pipeline bloqueado.")
            log.info("CAT-R04: " + str(len(df_raiz)) + " categoria(s) raiz encontrada(s).")
        else:
            df_raiz = df.copy()
            log.warning("CAT-R04: coluna id_categoria_pai ausente - usando todas as categorias.")

        # Seleciona colunas relevantes
        colunas = ["id_categoria", "nome_categoria", "bronze_ingested_at", "silver_processed_at"]
        colunas_presentes = [c for c in colunas if c in df_raiz.columns]
        df_raiz = df_raiz[colunas_presentes].reset_index(drop=True).copy()

        df_raiz["gold_processed_at"] = datetime.now()

        for col in df_raiz.columns:
            if pd.api.types.is_datetime64_any_dtype(df_raiz[col]):
                df_raiz[col] = df_raiz[col].dt.tz_localize(None)

        # Grava na Gold Delta
        write_deltalake(
            table_or_uri    = path_gold_cat,
            data            = df_raiz,
            mode            = "overwrite",
            storage_options = get_storage_options(),
            schema_mode     = "overwrite"
        )
        log.info("gold_categorias_raiz Delta gravado: " + str(len(df_raiz)) + " linhas.")

        # Grava no SQL Server
        df_spark = spark.createDataFrame(df_raiz)
        sucesso_sql = gravar_sql(df_spark, "gold_categorias_raiz")
        log.info("gold_categorias_raiz SQL Server: " + str("OK" if sucesso_sql else "FALHOU"))

        return True

    except Exception as e:
        log.error("Erro em gold_categorias_raiz: " + str(e))
        return False

### Validacao Pontual
Execute esta celula isoladamente para verificar o estado atual.

In [0]:
try:
    dep_ok = camada_anterior_concluida(CAMADA, TABELAS_SQUAD2)
    log.info("Silver CONCLUIDO: " + str("Sim" if dep_ok else "Aguardando"))

    visoes = ["gold_produtos_ativos", "gold_itens_pedido_validos", "gold_categorias_raiz"]
    for visao in visoes:
        try:
            dt    = DeltaTable(get_delta_path(CAMADA, visao), storage_options=get_storage_options())
            count = dt.to_pyarrow_dataset().count_rows()
            log.info(visao + ": " + str(count) + " linhas")
        except Exception:
            log.info(visao + ": ainda nao existe")

except Exception as e:
    log.error("Erro na validacao: " + str(e))
    raise

### Execucao Direta
Executada pelo Databricks Job apos Silver concluida.

In [0]:
if not camada_anterior_concluida(CAMADA, TABELAS_SQUAD2):
    log.warning("Silver nao concluida - encerrando Gold.")
else:
    log.info("Silver concluida - iniciando processamento Gold.")

    resultados = {}

    resultados["gold_produtos_ativos"]       = processar_gold_produtos()
    resultados["gold_itens_pedido_validos"]  = processar_gold_itens_pedido()
    resultados["gold_categorias_raiz"]       = processar_gold_categorias()

    log.info("=" * 50)
    log.info("RESUMO GOLD")
    log.info("=" * 50)
    for visao, sucesso in resultados.items():
        log.info(visao + ": " + str("OK" if sucesso else "FALHOU"))

    if all(resultados.values()):
        salvar_checkpoint(CAMADA, "gold", set(["gold_processado"]), status="CONCLUIDO")
        log.info("Gold concluida com sucesso.")
    else:
        log.error("Gold concluida com falhas.")

log_fim("feat_squad2_" + CAMADA, inicio)